# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 09: Model Evaluation & Multi-Policy Benchmarking

---

### 📖 Objective:
Compare the trained **RL Policy** against 3 pedagogical baselines across 250 simulated student episodes:
1. **RL Policy (Q-Learning)**
2. **Rule-Based Pedagogical Heuristic**
3. **Fixed Difficulty (Level 3)**
4. **Random Policy Baseline**


### Cell 1: Import Libraries & Baseline Agents


In [ ]:
# Cell 1: Imports & Baseline Definitions
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")

class RuleBasedAgent:
    def select_action(self, mastery: float) -> int:
        if mastery < 0.40: return np.random.choice([0, 1])
        elif mastery <= 0.70: return 2
        else: return np.random.choice([3, 4])

class StudentSimulator:
    def __init__(self): self.skill = np.random.beta(2, 3)
    def step(self, diff):
        diff_norm = (diff - 1) / 4.0
        prob = 1.0 / (1.0 + np.exp(-4.0 * (self.skill - diff_norm)))
        corr = int(np.random.rand() < prob)
        gain = 0.07 * (1.2 if corr else 0.8) if abs(self.skill - diff_norm) < 0.25 else 0.02
        self.skill = min(1.0, self.skill + gain)
        reward = 1.6 * gain + 0.4 * corr + 0.4 - 0.25 * ((self.skill - diff_norm)**2)
        return corr, gain, reward

class BKTTracker:
    def __init__(self): self.p_l = 0.25
    def update(self, corr):
        num = self.p_l * (0.9 if corr else 0.1)
        den = num + (1.0 - self.p_l) * (0.2 if corr else 0.8)
        self.p_l = float(np.clip((num / max(1e-6, den)) + 0.15, 0.01, 0.99))
        return self.p_l


### Cell 2: Run Comparative Simulation across 250 Episodes


In [ ]:
# Cell 2: 250-Episode Benchmark Simulation
q_table_learned = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [5.6, 2.9, 2.9, 2.1, 2.7],
    [5.2, 5.8, 5.2, 4.8, 4.5],
    [3.4, 5.5, 1.4, 3.1, 2.5],
    [5.6, 0.0, 1.7, 1.5, 0.0],
    [4.3, 5.6, 4.6, 4.7, 4.5],
    [5.6, 4.2, 5.8, 5.5, 5.6],
    [4.3, 3.4, 5.8, 1.5, 5.6],
    [5.6, 4.3, 4.8, 5.5, 5.3],
    [5.6, 5.6, 5.6, 5.8, 5.6]
])

policies = {
    "RL Policy (Q-Learning)": lambda s: int(np.argmax(q_table_learned[min(9, int(s * 10))])),
    "Rule-Based Heuristic": lambda s: RuleBasedAgent().select_action(s),
    "Fixed Difficulty (Lvl 3)": lambda s: 2,
    "Random Policy Baseline": lambda s: np.random.randint(0, 5)
}

results = {name: {"rewards": [], "gains": [], "zpd": []} for name in policies}

for name, policy_fn in policies.items():
    for _ in range(250):
        student, bkt = StudentSimulator(), BKTTracker()
        s_init = student.skill
        ep_r, zpd_hits = 0.0, 0
        for _ in range(20):
            a = policy_fn(bkt.p_l)
            if abs(student.skill - (a/4.0)) < 0.25: zpd_hits += 1
            corr, gain, r = student.step(a + 1)
            bkt.update(corr)
            ep_r += r
        results[name]["rewards"].append(ep_r)
        results[name]["gains"].append(student.skill - s_init)
        results[name]["zpd"].append(zpd_hits / 20.0)

summary_df = pd.DataFrame([{
    "Policy Algorithm": k,
    "Mean Cumulative Reward": round(np.mean(v["rewards"]), 3),
    "Avg Skill Gain (Δθ)": round(np.mean(v["gains"]), 4),
    "ZPD Alignment (%)": f"{np.mean(v['zpd'])*100:.1f}%"
} for k, v in results.items()])

print("--- Benchmark Performance Summary: ---")
display(summary_df)


### Cell 3: Plot Benchmark Comparison Charts


In [ ]:
# Cell 3: Plot Boxplots & Bar Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

labels = [k.replace(" Policy", "").replace(" Baseline", "") for k in policies.keys()]
axes[0].boxplot([results[k]["gains"] for k in policies], tick_labels=labels, patch_artist=True)
axes[0].set_title('Student Skill Gain Distribution (Δθ)', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].bar(labels, [np.mean(results[k]["rewards"]) for k in policies], color=['#10b981', '#6366f1', '#f59e0b', '#f43f5e'], alpha=0.85)
axes[1].set_title('Mean Cumulative Reward by Policy', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

axes[2].bar(labels, [np.mean(results[k]["zpd"])*100 for k in policies], color='#06b6d4', alpha=0.85)
axes[2].set_title('Zone of Proximal Development (ZPD) %', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 100)
axes[2].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print("✅ Step 09 Completed Successfully!")
